In [257]:
%reset -f
%reload_ext autoreload
%autoreload 2

In [258]:
import sys,os
import numpy as np
import pandas as pd
import re
from new_tools import iv_report
from new_tools import new_psi,feature_plot_generator
import warnings
warnings.filterwarnings("ignore")

In [259]:
sys.path.append("D:/work/scorecard")  
DATA_OR_PATH = 'D:/work/04.test/04-印尼（融360）/data_or/'

In [260]:
data_lst = ['Portrait', 'SMS', 'Telco']

In [261]:
data = data_lst[0]

In [262]:
data_path = f'D:/work/04.test/04-印尼（融360）/{data}/data/data_model.parquet'
data_all = pd.read_parquet(data_path)
df_list = []
temp_df = pd.DataFrame({
    'dataset':'all',
    'count':data_all['label'].count(),
    'badrate':data_all['label'].mean(),
    'badrate_weight':(data_all['label']*data_all['weight']).sum()/data_all['weight'].sum()
},index=[0])
df_list.append(temp_df)

def group_weight_badrate_cal(group):
    return (group['label']*group['weight']).sum()/group['weight'].sum()

for col in ['month_time']:
    temp_df = data_all.groupby(col).agg({'label':['count','mean']}).reset_index()
    temp_df.columns = ['dataset','count','badrate']
    temp_df['badrate_weight'] = (
        data_all.groupby(col)
        .apply(group_weight_badrate_cal)  # 使用apply接收完整分组
        .reset_index(drop=True)
    )
    temp_df.sort_values(by='dataset',ascending=True,inplace=True)
    df_list.append(temp_df)
info_df = pd.concat(df_list,axis=0,ignore_index=True)
info_df.drop(columns = ['badrate_weight'],axis=0,inplace=True)

In [263]:
info_df

,dataset,count,badrate
0,all,10001,0.227977
1,2025-03,1198,0.221202
2,2025-04,832,0.224760
3,2025-05,732,0.147541
4,2025-06,3011,0.251411
5,2025-07,4228,0.227767


In [264]:
with pd.ExcelWriter('特征评估报告.xlsx',mode='w') as writer:
    info_df.to_excel(writer, sheet_name='数据概要', index=False)

In [265]:
# std_rate = data_all[ft_lst].apply(lambda x: np.std(x), axis=0).reset_index()
# std_rate.rename(columns={"index": "var_name", 0: "std"}, inplace=True)

# the_min = data_all[ft_lst].apply(lambda x: np.min(x), axis=0).reset_index()
# the_min.rename(columns={"index": "var_name", 0: "min"}, inplace=True)

# the_max = data_all[ft_lst].apply(lambda x: np.max(x), axis=0).reset_index()
# the_max.rename(columns={"index": "var_name", 0: "max"}, inplace=True)

# basic_df = pd.merge(null_rate, std_rate, on="var_name")
# basic_df = pd.merge(basic_df, the_min, on="var_name")
# basic_df = pd.merge(basic_df, the_max, on="var_name")
# basic_df = 

In [266]:
for data in data_lst:

    print('**'*20,data,'**'*20)
    data_path = f'D:/work/04.test/04-印尼（融360）/{data}/data/data_model.parquet'
    data_all = pd.read_parquet(data_path)
    data_all['target'] = 'train'

    ex_lst = ['name','id-number','phone','time','due_date','label','overdue_days',
    'weight',
    'target',
    'month_time',
    'day_time',
    'week_time',]
    ft_lst = [i for i in data_all.columns if i not in ex_lst]
    len(ft_lst)

    null_rate=data_all[ft_lst].apply(lambda x: x.isnull().mean(), axis=0).reset_index()
    null_rate.rename(columns={"index": "var_name", 0: "missing_rate"}, inplace=True)
    basic_df = null_rate
    
    data_all.fillna(-999,inplace=True)

    iv_calculator = iv_report.IVCalculator(data_all,label='label',target='target',keep_list=ft_lst, max_leaf_nodes=6, min_samples_leaf=0.05)
    iv_df = iv_calculator.iv_report(use_thread=True,max_workers=20)
    iv_df.sort_values(by='train_iv',ascending=False,inplace=True)

    iv_detail_df = iv_calculator.detail_report(iv_df['var_name'],'all')

    def calc_lift(df):
        total_bad = df['bad'].sum()
        total_cnt = df['total'].sum()
        overall_bad_rate = total_bad / total_cnt
        df['lift'] = df['bad_rate'] / overall_bad_rate
        
        return df

    iv_detail_df = iv_detail_df.groupby('var_name', group_keys=False).apply(calc_lift)



    def extract_left(bin_str):
        """从区间字符串中提取左端点数值"""
        try:
            left = re.findall(r'[\[\(](-inf|-?\d+\.?\d*)', str(bin_str))[0]
            return float('-inf') if left == '-inf' else float(left)
        except IndexError:
            return float('inf')  # 异常值放最后

    iv_detail_df = iv_detail_df.sort_values(
        by=['var_name', 'bin'],
        key=lambda s: s if s.name == 'var_name' else s.map(extract_left),
        ascending=[True, True]  # var_name升序，bin升序
    )

    max_lift_df = iv_detail_df.groupby(['var_name'])['lift'].max().reset_index()
    max_lift_df.rename(columns = {'lift':'max_lift'},inplace=True)

    iv_df.rename(columns = {'train_iv':'total_iv'},inplace=True)

    iv_df = pd.merge(basic_df, iv_df, on="var_name", how="left")
    iv_df = pd.merge(iv_df,iv_detail_df[['var_name','ks']].drop_duplicates(['var_name','ks']) , on="var_name", how="left")
    iv_df = pd.merge(iv_df, max_lift_df, on="var_name", how="left")
    iv_df.fillna(0,inplace=True)
    with pd.ExcelWriter('特征评估报告.xlsx',mode='a') as writer:
        iv_df.to_excel(writer, sheet_name=f'{data}_关键特征', index=False)
        iv_detail_df.to_excel(writer, sheet_name=f'{data}_特征分箱', index=False)
        feature_plot_generator.create_feature_plot_report(
            excel_writer=writer,
            data=data_all,
            feature_list=ft_lst,
            sheet_name=f'{data}_特征分箱图',
            if_all = True
        )

**************************************** Portrait ****************************************
初始化IV计算器,获取各个数据集的索引...
初始化完成


计算在所有数据集上的IV分箱细节: 100%|██████████| 716/716 [00:02<00:00, 315.87it/s]


--- 正在向工作表 'Portrait_特征分箱图' 添加特征分箱图 ---
Step 1/4: 在训练集上计算特征IV值和分箱...


计算IV和分箱: 100%|██████████| 716/716 [00:08<00:00, 81.52it/s]


Step 2/4: 在Excel中创建工作表 'Portrait_特征分箱图'...
Step 3/4: 写入特征信息...
Step 4/4: 生成并插入特征分箱图...


...

--- 正在向工作表 'Telco_特征分箱图' 添加特征分箱图 ---
Step 1/4: 在训练集上计算特征IV值和分箱...


计算IV和分箱: 100%|██████████| 262/262 [00:03<00:00, 69.81it/s]


Step 2/4: 在Excel中创建工作表 'Telco_特征分箱图'...
Step 3/4: 写入特征信息...
Step 4/4: 生成并插入特征分箱图...


生成图表: 100%|██████████| 262/262 [00:59<00:00,  4.41it/s]


--- 特征分箱图已成功添加至工作表 'Telco_特征分箱图' ---
